# Writing Project Tools - Google Colab Template

Run this notebook in Google Colab to clone the toolkit from GitHub, install it in the runtime, scaffold a writing-project workspace, and generate Word documents from Markdown.

Default GitHub repo: `https://github.com/ruchirlives/src.git`

Use `Runtime > Run all` for a fresh setup. Colab runtimes are temporary unless you mount Google Drive and place the project workspace there.

## 1. Settings

Edit these values before running if you want a different branch, repository URL, or project folder name.

In [3]:
from pathlib import Path

REPO_URL = "https://github.com/ruchirlives/src.git"
REPO_BRANCH = "main"
TOOLKIT_DIR = Path("/content/writing-project-tools")
PROJECT_DIR = Path("/content/writing-project")

print(f"Toolkit repo: {REPO_URL}")
print(f"Toolkit path: {TOOLKIT_DIR}")
print(f"Writing project path: {PROJECT_DIR}")

Toolkit repo: https://github.com/ruchirlives/src.git
Toolkit path: /content/writing-project-tools
Writing project path: /content/writing-project


## 2. Optional: Use Google Drive For Persistent Work

Run this cell if you want the scaffolded article folder to persist after the Colab runtime shuts down. Skip it if `/content/writing-project` is fine for temporary work.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# @title Default title text
from pathlib import Path
from google.colab import drive

folder_name = "" # @param {"type":"string","placeholder":"writing-project"}
USE_GOOGLE_DRIVE = True # @param {"type":"boolean","placeholder":"False"}

# Use folder_name form field:

if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")

    # Determine the sub-folder name. If 'folder_name' is empty, default to "writing-project".
    project_sub_folder = folder_name if folder_name else "writing-project"

    # Construct the full project path within Google Drive
    PROJECT_DIR = Path("/content/drive/MyDrive/") / project_sub_folder
    print(f"Using persistent project path: {PROJECT_DIR}")
else:
    # PROJECT_DIR retains its value from the initial setup if USE_GOOGLE_DRIVE is False.
    print(f"Using temporary project path: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using persistent project path: /content/drive/MyDrive/volunteerReview


## 3. Clone Or Update The Toolkit Repo

This fetches the GitHub repository into the Colab runtime. Re-run after pushing changes to GitHub.

In [6]:
import subprocess

def run(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

if TOOLKIT_DIR.exists():
    run(["git", "fetch", "origin"], cwd=TOOLKIT_DIR)
    run(["git", "checkout", REPO_BRANCH], cwd=TOOLKIT_DIR)
    run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=TOOLKIT_DIR)
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(TOOLKIT_DIR)])

$ git clone --branch main https://github.com/ruchirlives/src.git /content/writing-project-tools


## 4. Install The Toolkit

Editable install means notebook changes can use the cloned source immediately.

In [7]:
import sys

run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-e", str(TOOLKIT_DIR)])

$ /usr/bin/python3 -m pip install --upgrade pip
$ /usr/bin/python3 -m pip install -e /content/writing-project-tools


## 5. Scaffold A Writing Project Workspace

This creates the Markdown source files, `AGENTS.md`, `assertions.csv`, `docx_sources.csv`, and project README. Existing files are left alone unless `FORCE_SCAFFOLD` is set to `True`.

In [13]:
FORCE_SCAFFOLD = False

command = [sys.executable, str(TOOLKIT_DIR / "scaffold_article.py"), str(PROJECT_DIR)]
if FORCE_SCAFFOLD:
    command.append("--force")

run(command)
print("\nProject files:")
for path in sorted(PROJECT_DIR.iterdir()):
    print("-", path.name)

$ /usr/bin/python3 /content/writing-project-tools/scaffold_article.py /content/drive/MyDrive/volunteerReview

Project files:
- .gitignore
- AGENTS.md
- README.md
- article-outline.md
- article-plan.md
- assertions.csv
- colleagues.md
- considerations.md
- context.md
- docx_sources.csv
- instructions_for_authors.md
- source-notes.md


## 6. Edit Project Markdown In Colab

Use the file browser on the left, or edit from cells like this. The scaffold creates `article-plan.md`, `article-outline.md`, source notes, context, considerations, colleagues, and author instructions.

In [9]:
plan_path = PROJECT_DIR / "article-plan.md"
print(plan_path.read_text(encoding="utf-8"))

# Article Plan For Comment

## Working Title


## Proposed Angle


## Core Message


## Proposed Content


## Voice And Scope


## Input Requested




## 7. Generate Word Documents

The output paths are controlled by `docx_sources.csv`. By default, generated `.docx` files go into `docs/` inside the project folder.

In [10]:
run(["create-writing-docs"], cwd=PROJECT_DIR)

docs_dir = PROJECT_DIR / "docs"
print("\nGenerated documents:")
for path in sorted(docs_dir.glob("*.docx")):
    print(path)

$ create-writing-docs

Generated documents:
/content/drive/MyDrive/communityEngagement/docs/article-outline.docx
/content/drive/MyDrive/communityEngagement/docs/article-plan.docx


## 8. Download Generated Files

Use this to download generated `.docx` files from the Colab runtime.

In [ ]:
from google.colab import files

for path in sorted((PROJECT_DIR / "docs").glob("*.docx")):
    files.download(str(path))

## 9. Project Editor In Colab

The local `edit-assertions` web server includes both assertions review and Markdown editing. In Colab, run it with a public port proxy URL using `google.colab.output.eval_js`; open `/markdown` on that proxy URL to edit project `.md` files.

In [19]:
import os
import signal
import subprocess
import time
from google.colab import output

PORT = 8765
ASSERTIONS_PROCESS = None

if ASSERTIONS_PROCESS is not None and ASSERTIONS_PROCESS.poll() is None:
    os.kill(ASSERTIONS_PROCESS.pid, signal.SIGTERM)

ASSERTIONS_PROCESS = subprocess.Popen(
    ["edit-assertions", "--host", "0.0.0.0", "--port", str(PORT), "--no-open"],
    cwd=PROJECT_DIR,
)
time.sleep(1)

print("Project editor URL:")
print(output.eval_js(f"google.colab.kernel.proxyPort({PORT})"))
print("Open /markdown on that URL for the Markdown editor.")
print("Stop the server with the next cell when finished.")

Project editor URL:
https://8765-m-s-kkb-usc1a0-147gq1mug7s52-a.us-central1-0.prod.colab.dev
Open /markdown on that URL for the Markdown editor.
Stop the server with the next cell when finished.


In [17]:
if 'ASSERTIONS_PROCESS' in globals() and ASSERTIONS_PROCESS is not None and ASSERTIONS_PROCESS.poll() is None:
    ASSERTIONS_PROCESS.terminate()
    print("Assertions editor stopped.")
else:
    print("Assertions editor is not running.")

Assertions editor is not running.


In [18]:
# We need to delete any old assertions processes as well
!kill -9 $(ps aux | grep edit-assertions | awk '{print $2}')


^C
